# AI-Based Robotic Fruit Sorting System
## GDRIVE CURATED TRAINING NOTEBOOK
This notebook mounts your Google Drive, unzips `MAIN-EL-IOT/CURATED_DATASET.zip` directly to the fast local Colab disk, trains the model, and automatically saves the trained `.keras` and `.tflite` files back to your Drive.

### SECTION 1 — Environment Setup & Google Drive Mount

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import matplotlib.pyplot as plt
import random

# Fix Seeds
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Environment Ready and Drive Mounted!")

### SECTION 2 — Extract Curated Dataset

In [ ]:
import zipfile
import shutil

ZIP_PATH = '/content/drive/MyDrive/MAIN-EL-IOT/CURATED_DATASET.zip'
LOCAL_DATA_DIR = '/content/dataset'

if os.path.exists(LOCAL_DATA_DIR):
    shutil.rmtree(LOCAL_DATA_DIR)
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

print(f"⏳ Extracting {ZIP_PATH} to local storage (this is very fast)...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA_DIR)
    print("✨ Extraction Complete!")
except Exception as e:
    print(f"❌ Error extracting dataset. Make sure the file exists at {ZIP_PATH}.")
    print(e)

### SECTION 3 — Locate Dataset Root & Verify Classes

In [ ]:
def find_dataset_root(root_path):
    # Automatically find the folder that actually contains the class folders
    # (In case the zip extracted into CURATED_DATASET/CURATED_DATASET/...)
    for root, dirs, files in os.walk(root_path):
        if any("Apple" in d or "Grape" in d or "Strawberry" in d for d in dirs):
            return root
    return root_path

ACTUAL_DATASET_DIR = find_dataset_root(LOCAL_DATA_DIR)
print(f"📂 Dataset root located at: {ACTUAL_DATASET_DIR}")

# Print Image Counts
print('\nImage counts per class:')
for cls in os.listdir(ACTUAL_DATASET_DIR):
    cls_path = os.path.join(ACTUAL_DATASET_DIR, cls)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f'  {cls}: {count} images')

### SECTION 4 — Train / Validation Setup

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    ACTUAL_DATASET_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    ACTUAL_DATASET_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

ALLOWED_CLASSES = train_ds.class_names
print('\n✅ Classes Detected for Training:', ALLOWED_CLASSES)

### SECTION 5 — Data Augmentation & Pipeline

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2)
])

AUTOTUNE = tf.data.AUTOTUNE
def prepare_pipeline(ds, augment=False):
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.cache().prefetch(buffer_size=AUTOTUNE)

train_ds = prepare_pipeline(train_ds, augment=True)
val_ds = prepare_pipeline(val_ds)
print('Data pipeline with Augmentation is ready.')

### SECTION 6 — CNN Architecture

In [ ]:
model = Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(len(ALLOWED_CLASSES), activation='softmax')
])

model.summary()

### SECTION 7 — Compile & Train

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

# We will temporarily save the best model to Colab disk during training
checkpoint = tf.keras.callbacks.ModelCheckpoint('/content/best_model.keras', save_best_only=True)

print("\n🚀 Starting Training...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stopping, checkpoint]
)

### SECTION 8 — Visualize Accuracy

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')
plt.show()

### SECTION 8.5 — Advanced Evaluation Metrics
Generates the Confusion Matrix, Precision, Recall, and F1-Score.

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("\n🔍 Extracting Validation Data for Evaluation...")
val_images = []
val_labels = []
for images, labels in val_ds.unbatch():
    val_images.append(images.numpy())
    val_labels.append(labels.numpy())

val_images = np.array(val_images)
val_labels = np.array(val_labels)

print("🧠 Running Predictions...")
predictions = model.predict(val_images)
pred_labels = np.argmax(predictions, axis=1)

print("\n📊 Classification Report (Precision, Recall, F1-Score):")
print(classification_report(val_labels, pred_labels, target_names=ALLOWED_CLASSES))

# Confusion Matrix
cm = confusion_matrix(val_labels, pred_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=ALLOWED_CLASSES, yticklabels=ALLOWED_CLASSES)
plt.xlabel('Predicted Fruit')
plt.ylabel('Actual Fruit')
plt.title('Confusion Matrix Heatmap')
plt.show()


### SECTION 9 — Safe Export to Google Drive
This ensures your trained models are permanently saved to your Drive so you don't lose them when Colab disconnects.

In [ ]:
import json

DRIVE_EXPORT_DIR = '/content/drive/MyDrive/MAIN-EL-IOT/Models'
os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)

# 1. Save .keras
keras_path = os.path.join(DRIVE_EXPORT_DIR, 'final_model.keras')
model.save(keras_path)
print(f'✅ Saved: {keras_path}')

# 2. Save TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
tflite_path = os.path.join(DRIVE_EXPORT_DIR, 'model.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f'✅ Saved: {tflite_path}')

# 3. Save Class Names
json_path = os.path.join(DRIVE_EXPORT_DIR, 'class_names.json')
with open(json_path, 'w') as f:
    json.dump(ALLOWED_CLASSES, f)
print(f'✅ Saved: {json_path}')

print('\n🎉 All models successfully backed up to Google Drive!')

### SECTION 10 — Inference / ESP32 Test Hooks (Optional)

In [ ]:
import requests

ESP32_IP = '192.168.4.1'

def trigger_sorting_arm(predicted_class):
    if 'Fresh' in predicted_class:
        angles = {'base': 120, 'shoulder': 90, 'elbow': 45, 'claw': 10}
    elif 'Rotten' in predicted_class:
        angles = {'base': 60, 'shoulder': 90, 'elbow': 45, 'claw': 10}
    else:
        print('Unknown fruit - no action.')
        return
        
    url = f"http://{ESP32_IP}/move?base={angles['base']}&shoulder={angles['shoulder']}&elbow={angles['elbow']}&claw={angles['claw']}"
    print(f'Sending to ESP32: {url}')
    # requests.get(url, timeout=2) # Uncomment when connected to ESP32 network

print("ESP32 Hook Ready!")